# Notebook 3 — RNN: Feature Extraction, Preprocessing & Training

## 1 Setup

In [1]:
from pathlib import Path
import json, random, sys, time

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.wajib.rnn.RNN import buildRNNKeras, trainRNNKeras, trainRNNDataset, RNNScratch
from src.wajib.shared.layers import EmbeddingLayer, DenseLayer
from src.wajib.shared.preprocessing import (
    loadFlickr8kCaptions, buildVocabulary, saveVocabulary, loadVocabulary,
)
from src.wajib.shared.decoder import greedyDecode
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs:", gpus)

I0000 00:00:1778842033.317972   35043 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778842033.328789   35043 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778842035.283135   35043 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778842043.047523   35043 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_E

GPUs: []


E0000 00:00:1778842046.723302   35043 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 2 Config

In [2]:
FEATURES_NPY  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_features.npy"
FEATURES_IDX  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_index.json"
CAPTIONS_FILE = PROJECT_ROOT / "data/flickr8k/captions.txt"
VOCAB_PATH    = PROJECT_ROOT / "src/wajib/weights/vocab.json"
WEIGHTS_DIR   = PROJECT_ROOT / "src/wajib/weights/rnn"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

EMBED_DIM    = 256
MAX_LEN      = 30
EPOCHS       = 30
BATCH_SIZE   = 64
CNN_FEAT_DIM = 2048

# 3 variasi num_layers × 2 variasi hidden_dim = 6 total
VARIATIONS = [
    (1, 128),
    (1, 512),
    (2, 128),
    (2, 512),
    (3, 128),
    (3, 512),
]

### 3 Load CNN Features

In [3]:
features_matrix = np.load(FEATURES_NPY)
with open(FEATURES_IDX) as f:
    idx_names = json.load(f)

image_features = {name: features_matrix[i] for i, name in enumerate(idx_names)}
print(f"Loaded {len(image_features)} features, dim={features_matrix.shape[1]}")

Loaded 8091 features, dim=2048


### 4 Load Captions + Split + Vocab

In [4]:
captions_dict = loadFlickr8kCaptions(str(CAPTIONS_FILE))

all_images = list(captions_dict.keys())
random.shuffle(all_images)  # seed already set in cell 1

train_imgs = set(all_images[:6000])
val_imgs   = set(all_images[6000:7000])
test_imgs  = set(all_images[7000:])

train_caps = {k: v for k, v in captions_dict.items() if k in train_imgs}
val_caps   = {k: v for k, v in captions_dict.items() if k in val_imgs}
test_caps  = {k: v for k, v in captions_dict.items() if k in test_imgs}

print(f"Split => train={len(train_caps)}, val={len(val_caps)}, test={len(test_caps)}")

if VOCAB_PATH.exists():
    vocab = loadVocabulary(str(VOCAB_PATH))
    print(f"Vocab loaded: {len(vocab)}")
else:
    all_train_caps = [cap for caps in train_caps.values() for cap in caps]
    vocab = buildVocabulary(all_train_caps, min_freq=2)
    saveVocabulary(vocab, str(VOCAB_PATH))
    print(f"Vocab built + saved: {len(vocab)}")

Split => train=6000, val=1000, test=1091
Vocab loaded: 4558


### 5 Build Dataset Arrays

In [5]:
X_cnn_tr, X_tok_tr, y_tr = trainRNNDataset(image_features, train_caps, vocab, MAX_LEN)
X_cnn_va, X_tok_va, y_va = trainRNNDataset(image_features, val_caps,   vocab, MAX_LEN)

print(f"Train -> X_cnn={X_cnn_tr.shape}, X_tok={X_tok_tr.shape}, y={y_tr.shape}")
print(f"Val   -> X_cnn={X_cnn_va.shape}, X_tok={X_tok_va.shape}, y={y_va.shape}")

Train -> X_cnn=(30000, 2048), X_tok=(30000, 30), y=(30000, 31)
Val   -> X_cnn=(5000, 2048), X_tok=(5000, 30), y=(5000, 31)


### 6 Training (6 Variation)

In [8]:
histories = {}

for i, (num_layers, hidden_dim) in enumerate(VARIATIONS):
    name      = f"rnn_{num_layers}L_{hidden_dim}h"
    save_path = WEIGHTS_DIR / f"{name}.keras"

    print(f"\n{'='*50}\n  {name}\n{'='*50}")

    model = buildRNNKeras(
        vocab_size     = len(vocab),
        embed_dim      = EMBED_DIM,
        hidden_dim     = hidden_dim,
        num_rnn_layers = num_layers,
        cnn_feature_dim= CNN_FEAT_DIM,
        rnn_type = 'rnn'
    )
    if i == 0:
        model.summary()

    hist = trainRNNKeras(
        model,
        X_cnn_tr, X_tok_tr, y_tr,
        X_cnn_va, X_tok_va, y_va,
        epochs     = EPOCHS,
        batch_size = BATCH_SIZE,
        save_path  = str(save_path),
    )
    histories[name] = hist
    print(f"  Saved to {save_path.name}")

print(f"\nSemua {len(VARIATIONS)} variasi selesai.")


  rnn_1L_128h


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cnn_feature         │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_ids           │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_proj (Dense)    │ (None, 256)       │    524,544 │ cnn_feature[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  1,166,848 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ token_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 256)    │          0 │ cnn_proj[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims_1       │ (None, None, 1)   │          0 │ not_equal_1[0][0] │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zeros_like_1        │ (None, None, 256) │          0 │ embedding[0][0]   │
│ (ZerosLike)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ones_like_1         │ (None, 1, 256)    │          0 │ reshape_1[0][0]   │
│ (OnesLike)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logical_or_1        │ (None, None, 256) │          0 │ expand_dims_1[0]… │
│ (LogicalOr)         │                   │            │ zeros_like_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, None, 256) │          0 │ ones_like_1[0][0… │
│ (Concatenate)       │                   │            │ logical_or_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, None, 256) │          0 │ reshape_1[0][0],  │
│ (Concatenate)       │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_1 (Any)         │ (None, None)      │          0 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_0 (SimpleRNN)   │ (None, None, 128) │     49,280 │ concatenate_2[0]… │
│                     │                   │            │ any_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, None, 128) │          0 │ rnn_0[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, None,      │    587,982 │ dropout_1[0][0]   │
│                     │ 4558)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,328,654 (8.88 MB)

 Trainable params: 2,328,654 (8.88 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - loss: 4.3213 - val_loss: 3.5743
Epoch 2/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 89ms/step - loss: 3.4700 - val_loss: 3.2929
Epoch 3/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 45s 96ms/step - loss: 3.2553 - val_loss: 3.1442
Epoch 4/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 88ms/step - loss: 3.1111 - val_loss: 3.0625
Epoch 5/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 89ms/step - loss: 3.0175 - val_loss: 3.0065
Epoch 6/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 89ms/step - loss: 2.9398 - val_loss: 2.9634
Epoch 7/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 90ms/step - loss: 2.8718 - val_loss: 2.9251
Epoch 8/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 90ms/step - loss: 2.8156 - val_loss: 2.9013
Epoch 9/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 43s 92ms/step - loss: 2.7686 - val_loss: 2.8837
Epoch 10/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 88ms/step - loss: 2.7269 - val_loss: 2.8729
Epoch 11/30
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 88ms/step - loss: 2.6887 - val_loss: 2.8595
Epoch 12/30
469/469 ━━━━━━━━━━

KeyboardInterrupt: 

### Training and Loss Curve

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (name, hist) in zip(axes.flatten(), histories.items()):
    ax.plot(hist['loss'],     label='train')
    ax.plot(hist['val_loss'], label='val')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

plt.suptitle("RNN Training Loss per Variasi", y=1.01)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / "rnn_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

### Training Summary

In [ ]:
print(f"{'Model':<22} {'Best Val Loss':>14} {'Epochs':>7}")
print('-' * 45)
for name, hist in histories.items():
    print(f"{name:<22} {min(hist['val_loss']):>14.4f} {len(hist['val_loss']):>7}")

best_name   = min(histories, key=lambda n: min(histories[n]['val_loss']))
best_layers = int(best_name.split('L')[0].split('_')[1])
print(f"\nBest: {best_name}")